# Exercise 10: Correcting chromaticity with sextupoles

**Block 3: Dispersion and chromaticity**

**Learning objectives**
* Understand why a sextupole at a place with dispersion acts like an extra quadrupole of strength
  $k_2LD\delta$ for off-momentum particles.
* Set the two sextupole families SF and SD by solving a 2x2 linear system so that $Q_x' = Q_y' = 0$.
* See the price: sextupoles are nonlinear, they distort phase space and limit the **dynamic aperture**.

**Motivation.** Chromaticity spreads the tunes of a beam with a momentum spread, and (in real machines)
a negative chromaticity drives the head–tail instability. We want to choose $Q'$. Quadrupoles cannot
do it (they focus all momenta the same way), but a sextupole can, if it sits where off-momentum
particles are displaced, i.e. where $D\neq0$.

> **Student version.** Fill in the cells marked `# YOUR CODE HERE` and run the check cells after them: they tell you when you are right.

In [ ]:
# --- setup: make `beamlib` importable --------------------------------------
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "beamlib").is_dir())
sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt
from beamlib import elements as E, optics as O, lattices as L, tracking as T, diagnostics as D, plotting as P
from beamlib.plotting import COLORS, MM
P.setup()

ring0 = L.fodo_ring()                          # sextupoles present but switched off (k2 = 0)
print([el.name for el in ring0 if isinstance(el, E.ThinMultipole)])

## 1. A sextupole seen by an off-momentum particle

An off-momentum particle oscillates around $x = D\delta$. Write its position as $x = D\delta + u$,
where $u$ is the betatron part. The sextupole kick is

$$\Delta p_x = -\frac{k_2L}{2}x^2 = -\frac{k_2L}{2}\left(D^2\delta^2 + 2D\delta\,u + u^2\right).$$

The middle term is linear in $u$: exactly the kick of a thin quadrupole with $kl = k_2L D \delta$.
(The first term is a small constant kick, the last one the nonlinear part.) In $y$ the effective
quad has the opposite sign.

**Task.** Verify this numerically: compute the derivative $\partial(\Delta p_x)/\partial u$ of `E.thin_multipole` at
$x = D\delta$ with a finite difference, and compare with $−k_2L D \delta$.

In [ ]:
def effective_quad_kl(k2l: float, D: float, delta: float) -> float:
    """Integrated quad strength seen by an off-momentum particle in a thin sextupole."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")


k2l, Dsx, dl, h = 3.0, 2.5, 1e-3, 1e-7
p = np.array([[Dsx * dl + h, Dsx * dl - h], [0, 0], [0, 0], [0, 0]])     # 4 rows: no 1/(1+delta) here
kick = E.thin_multipole(p, knl=[0, 0, k2l])[1, :]
slope = (kick[0] - kick[1]) / (2 * h)
print(f"d(dpx)/du = {slope:.6e},  -k2l*D*delta = {-effective_quad_kl(k2l, Dsx, dl):.6e}")
assert np.isclose(slope, -effective_quad_kl(k2l, Dsx, dl), rtol=1e-6)
print("GOOD! A sextupole at D != 0 is a delta-dependent quadrupole")

## 2. Optics at the sextupoles

`optics_at(ring, name)` (given) returns $\beta_x$, $\beta_y$ and $D$ at an element.

In [ ]:
def optics_at(ring, name):
    """beta_x, beta_y and D at the exit of the element called `name`."""
    tw = O.ring_twiss(ring, n_slices=1)
    i = [k for k, el in enumerate(ring) if el.name == name][0]
    j = np.where(tw["elem"] == i)[0][-1]
    return tw["betx"][j], tw["bety"][j], tw["dx"][j]

opt_sf, opt_sd = optics_at(ring0, "sf"), optics_at(ring0, "sd")
for nm, (bx, by, Dd) in [("SF", opt_sf), ("SD", opt_sd)]:
    print(f"{nm}: beta_x = {bx:.3f} m, beta_y = {by:.3f} m, beta_x/beta_y = {bx / by:.2f}, D = {Dd:.3f} m")

SF sits where $\beta_x \gg \beta_y$, SD where $\beta_y \gg \beta_x$, and both have $D > 0$. So SF mostly changes $Q_x'$
and SD mostly $Q_y'$.

## 3. The 2x2 system

Each sextupole adds $\Delta Q' = \pm\beta D k_2L / 4\pi$ (using $\Delta Q = \beta \Delta kl / 4\pi$ with $\Delta kl = k_2L D \delta$,
and the opposite sign in $y$):

$$\begin{pmatrix} Q'_x \\ Q'_y\end{pmatrix} = \begin{pmatrix} Q'_{x0} \\ Q'_{y0}\end{pmatrix} +
\frac{1}{4\pi}\begin{pmatrix} \beta_x^{SF}D^{SF} & \beta_x^{SD}D^{SD} \\ -\beta_y^{SF}D^{SF} & -\beta_y^{SD}D^{SD}\end{pmatrix}
\begin{pmatrix} k_2L^{SF} \\ k_2L^{SD}\end{pmatrix}$$

**Predict first.** Which sign do you expect for $k_2L$ of SF and of SD?

*Your prediction:*

**Task.** Write `chromaticity_correction` returning ($k_2L_\text{SF}$, $k_2L_\text{SD}$) for given target
chromaticities. Use `np.linalg.solve`.

In [ ]:
def natural_chromaticity_exact(ring):
    h = 1e-6
    qp = np.array(O.tunes_from_matrix(O.one_turn_matrix(ring, h)))
    qm = np.array(O.tunes_from_matrix(O.one_turn_matrix(ring, -h)))
    return (qp - qm) / (2 * h)


def chromaticity_correction(xi0, opt_sf, opt_sd, target=(0.0, 0.0)):
    """Integrated sextupole strengths (k2l_SF, k2l_SD) [1/m^2] giving the target Q'x, Q'y."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
xi0 = natural_chromaticity_exact(ring0)
k2_sf, k2_sd = chromaticity_correction(xi0, opt_sf, opt_sd)
print(f"natural chromaticity: {xi0.round(4)}")
print(f"SF: k2L = {k2_sf:.4f} 1/m^2,  SD: k2L = {k2_sd:.4f} 1/m^2")
ring = L.fodo_ring(k2_sf=k2_sf, k2_sd=k2_sd)

## 4. Check by tracking

With sextupoles the ring is nonlinear, so we cannot simply multiply matrices. Instead, for each $\delta$ we
find the closed orbit of the *tracking* map (Newton's method, `O.find_closed_orbit`), compute the
Jacobian (the matrix of the map linearised around that orbit, `O.numerical_jacobian`) and read the
tunes from it. This includes everything: chromatic quads, the sextupoles' feed-down, dispersion.

In [ ]:
def tunes_tracking(ring, delta):
    otm = T.make_one_turn_map(ring)
    co = O.find_closed_orbit(otm, delta)
    f4 = lambda b: otm(np.vstack([b, np.full((1, b.shape[1]), delta)]))[:4]
    return O.tunes_from_matrix(O.numerical_jacobian(f4, co))

deltas = np.linspace(-5e-3, 5e-3, 21)
q_before = np.array([tunes_tracking(ring0, d) for d in deltas])
q_after = np.array([tunes_tracking(ring, d) for d in deltas])
slope_after = np.polyfit(deltas, q_after, 1)[0]
print("Q' after correction (fit over +-5e-3):", slope_after.round(4))
h = 1e-4
xi_after = (np.array(tunes_tracking(ring, h)) - np.array(tunes_tracking(ring, -h))) / (2 * h)
print("Q' after correction (derivative at delta = 0):", xi_after.round(5))
assert np.all(np.abs(xi_after) < 0.02), "chromaticity should be ~0 after correction"
print("GOOD! Chromaticity corrected")

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, k, nm, col in [(axs[0], 0, "x", COLORS["x"]), (axs[1], 1, "y", COLORS["y"])]:
    ax.plot(deltas * 1e3, q_before[:, k], "o--", color=col, alpha=0.5, label="before")
    ax.plot(deltas * 1e3, q_after[:, k], "o-", color=col, label="after")
    ax.set_xlabel(r"$\delta$ [$10^{-3}$]"); ax.set_ylabel(f"$Q_{nm}$ (fractional)"); ax.legend()
plt.show()

The tune no longer changes linearly with $\delta$. What remains is a small parabola: the *second-order*
chromaticity, which two sextupole families cannot remove at the same time.

## 5. The price: nonlinear phase space

The sextupoles also kick on-momentum particles, with $\Delta p_x \propto x^2$. Small amplitudes do not notice,
large amplitudes do.

**Predict first.** Track on-momentum particles at increasing amplitude $x_0$ (up to 120 mm, a huge
value for this little ring) without sextupoles, with the sextupoles for $Q' = 0$, and with the
stronger ones needed for $Q' = +5$. What will the phase-space pictures look like?

*Your prediction:*

In [ ]:
# (about 5 s)
twx = O.twiss_from_matrix(O.one_turn_matrix(ring)[:2, :2])
s5 = chromaticity_correction(xi0, opt_sf, opt_sd, target=(5, 5))
ring5 = L.fodo_ring(k2_sf=s5[0], k2_sd=s5[1])
panels = [(ring0, "sextupoles off", 120e-3), (ring, r"sextupoles for $Q' = 0$", 120e-3),
          (ring5, r"stronger sextupoles, $Q' = +5$", 32e-3)]
fig, axs = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, (rg, title, amax) in zip(axs, panels):
    amps = np.linspace(amax / 16, amax, 16)
    bunch = np.zeros((5, len(amps))); bunch[0, :] = amps
    r = T.track(bunch, T.make_one_turn_map(rg), 400)
    P.poincare(ax, r.x[:, r.survived], r.px[:, r.survived], twx["beta"], twx["alpha"], s=2.0,
               values=amps[r.survived] * MM, cmap="viridis")
    ax.set_title(f"{title}\n{np.sum(~r.survived)} of {len(amps)} particles lost")
plt.show()

Without sextupoles all circles are perfect. With them, the large-amplitude curves are distorted,
and with stronger sextupoles they become clearly triangular (a sextupole has threefold symmetry,
Exercise 1); the outermost particles are lost. Note the different axis ranges.
The largest amplitude that survives is related to the **dynamic aperture** (DA).

## 6. Dynamic aperture vs chromaticity

Machines often run with *positive* chromaticity (to keep the beam stable). That needs stronger
sextupoles.

**Task.** Complete `dynamic_aperture(ring)`: track particles at amplitudes `amps` for 500 turns and
return the smallest amplitude that was lost (or `np.nan` if none was). Then the cell below scans
the target chromaticity.

In [ ]:
def dynamic_aperture(ring, amps=np.linspace(2e-3, 150e-3, 75), n_turns=500):
    """Smallest initial x (with px = y = py = 0, delta = 0) that is lost within n_turns [m]."""
    bunch = np.zeros((5, len(amps)))
    bunch[0, :] = amps
    bunch[2, :] = 1e-4                  # a tiny vertical offset, so y is not exactly zero
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
# (about 5 s)
targets = [0, 2, 5, 10, 20]
das, strengths = [], []
for tq in targets:
    s2 = chromaticity_correction(xi0, opt_sf, opt_sd, target=(tq, tq))
    strengths.append(s2)
    das.append(dynamic_aperture(L.fodo_ring(k2_sf=s2[0], k2_sd=s2[1])))
    print(f"Q' = {tq:3d}: k2L SF = {s2[0]:7.3f}, SD = {s2[1]:8.3f} 1/m^2 -> DA = {das[-1] * MM:6.1f} mm")
assert np.isnan(dynamic_aperture(ring0)), "without sextupoles the ring is linear: nothing is lost"
assert das[0] > das[-1], "stronger sextupoles should reduce the DA"

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(targets, np.array(das) * MM, "o-", color=COLORS["x"])
ax.set_xlabel(r"target chromaticity $Q_x' = Q_y'$"); ax.set_ylabel(r"dynamic aperture $x_0$ [mm]")
ax.set_title("Stronger sextupoles, smaller dynamic aperture")
plt.show()

Our toy ring has a small natural chromaticity and only needs weak sextupoles, so its DA is huge
(about $40\sigma$ for a 1 mm mrad beam, $σ\sigma_x$ = 2.8 mm here). In a large machine like the LHC the natural
chromaticity is $\approx−100$, the sextupoles are much stronger, and the dynamic aperture becomes one of
the main design constraints.

## Reflection questions

1. Why do we need *two* sextupole families, placed at locations with different $\beta_x/\beta_y$?
2. Could we correct chromaticity with sextupoles placed where $D = 0$?
3. The correction is linear (a 2x2 system), but the sextupoles are nonlinear magnets. Which
   effect does the linear calculation describe correctly, and which does it ignore?